
<style>
h2 {
  color: #FDB813;
  font-family: 'Segoe UI', 'Helvetica Neue', Arial, 'sans-serif';
  margin-bottom: 0.3em;
  border-left: 5px solid #FFDD57;
  padding-left: 10px;
}
p {
  font-size: 16px;
  color: #444451;
  font-family: 'Georgia', serif;
  margin-bottom: 0.6em;
}
hr {
  border: none;
  height: 2px;
  background: linear-gradient(to right, #FDB813, #fff 50%, #FDB813);
  margin: 20px 0;
}
ul {
  font-size: 15px;
  color: #323146;
  margin-left: 20px;
}
</style>

<br>

<h2>NYC YELLOW TAXI DATA PIPELINE &mdash; 2025</h2>
<hr/>

<p>
Efficient data engineering workflow to extract, load, and persist historical NYC yellow taxi trip records for 2025.  
Experience a streamlined, reliable, and robust pipeline.
</p>

<ul>
  <li><b>Extract:</b> Download monthly yellow taxi Parquet files from official sources</li>
  <li><b>Load:</b> Ingest curated Parquet files into Delta Lake bronze table</li>
  <li><b>Persist & Confirm:</b> Log successful loads for easy data governance</li>
</ul>

In [0]:
import urllib.request
import os
import shutil

#### EXTRACT ALL HISTORICAL YELLOW TAXI TRIP FOR YEAR - 2025 FROM API TO ADLS

In [0]:
months_to_process = ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', 
                     '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']

for months in months_to_process:
    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{months}.parquet'
    response = urllib.request.urlopen(url)

    dir_path = f'/Volumes/nyctaxi/landing/yellow_taxi/raw/2025/{months}'
    os.makedirs(dir_path, exist_ok = True)

    local_file_path = dir_path + f'/yellow_tripdata_{months}.parquet'
    with open(local_file_path, 'wb') as f:
        shutil.copyfileobj(response, f)

#### LOAD HISTORICAL YELLOW TAXI 
- `NYCTAXI.BRONZE.YELLOW_TAXI`

In [0]:
from pyspark.sql.functions import input_file_name, col, current_timestamp

yellow_taxi_historical_df = (spark.read.format('parquet')
                      .load('/Volumes/nyctaxi/landing/yellow_taxi/raw/2025/*')
                      .withColumn('file_name', col('_metadata.file_path'))
                      .withColumn('load_timestamp', current_timestamp())
                )
display(yellow_taxi_historical_df.limit(1))

In [0]:
# LOAD THE RESULTANT HISTORICAL DATA INTO TEMP VIEW
yellow_taxi_historical_df.createOrReplaceTempView('historical_yellow_taxi_df_temp_vw')

In [0]:
%skip
%sql
CREATE OR REPLACE TEMP VIEW yellow_taxi_historical_temp_vw_dedup AS
SELECT *
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, file_name
           ORDER BY load_time_stamp DESC
         ) AS rn
  FROM historical_yellow_taxi_df_temp_vw
)
WHERE rn = 1

#### LOAD HISTORICAL NYC TAXI TRIP
- `NYCTAXI.BRONZE.YELLOW_TAXI`

In [0]:
%sql
SELECT MAX(load_timestamp) AS max_load_ts
        FROM NYCTAXI.BRONZE.YELLOW_TAXI;

MERGE INTO NYCTAXI.BRONZE.YELLOW_TAXI tgt
USING (SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, file_name ORDER BY load_timestamp DESC) as rn FROM historical_yellow_taxi_df_temp_vw) WHERE rn = 1) src
ON tgt.VendorID = src.VendorID
   AND tgt.tpep_pickup_datetime = src.tpep_pickup_datetime
   AND tgt.tpep_dropoff_datetime = src.tpep_dropoff_datetime
   AND tgt.file_name = src.file_name

WHEN MATCHED THEN
  UPDATE SET
    passenger_count = src.passenger_count,
    trip_distance = src.trip_distance,
    RatecodeID = src.RatecodeID,
    store_and_fwd_flag = src.store_and_fwd_flag,
    PULocationID = src.PULocationID,
    DOLocationID = src.DOLocationID,
    payment_type = src.payment_type,
    fare_amount = src.fare_amount,
    extra = src.extra,
    mta_tax = src.mta_tax,
    tip_amount = src.tip_amount,
    tolls_amount = src.tolls_amount,
    improvement_surcharge = src.improvement_surcharge,
    total_amount = src.total_amount,
    congestion_surcharge = src.congestion_surcharge,
    Airport_fee = src.Airport_fee,
    cbd_congestion_fee = src.cbd_congestion_fee,
    file_name = src.file_name,
    load_timestamp = src.load_timestamp

WHEN NOT MATCHED THEN
  INSERT (
    VendorID,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    passenger_count,
    trip_distance,
    RatecodeID,
    store_and_fwd_flag,
    PULocationID,
    DOLocationID,
    payment_type,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    improvement_surcharge,
    total_amount,
    congestion_surcharge,
    Airport_fee,
    cbd_congestion_fee,
    file_name,
    load_timestamp
  )
  VALUES (
    src.VendorID,
    src.tpep_pickup_datetime,
    src.tpep_dropoff_datetime,
    src.passenger_count,
    src.trip_distance,
    src.RatecodeID,
    src.store_and_fwd_flag,
    src.PULocationID,
    src.DOLocationID,
    src.payment_type,
    src.fare_amount,
    src.extra,
    src.mta_tax,
    src.tip_amount,
    src.tolls_amount,
    src.improvement_surcharge,
    src.total_amount,
    src.congestion_surcharge,
    src.airport_fee,
    src.cbd_congestion_fee,
    src.file_name,
    src.load_timestamp
  );

In [0]:
dbutils.notebook.exit('HISTORICAL NYC YELLOW TRIP FILES HAS BEEN LOADED INTO NYCTAXI.BRONZE.YELLOW_TAXI')